# Week 58, Day 4: Boosting: AdaBoost and gradient boosting

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w058/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: AdaBoost from scratch

Complete weighted_stump (for each feature and threshold, the weighted error of predicting s on the right and -s on the left, for s = ±1) and the AdaBoost loop: the vote alpha and the weight update. Report the accuracy of one stump and of the boosted ensemble after 10, 50 and 200 rounds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(584)

def ring(n, g):
    X = g.uniform(-1, 1, (n, 2))
    y = np.where(X[:, 0] ** 2 + X[:, 1] ** 2 < 0.45, 1.0, -1.0)
    return X, np.where(g.random(n) < 0.05, -y, y)

def weighted_stump(X, y, w):
    best = (0, 0.0, 1, np.inf)
    for j in range(X.shape[1]):
        o = np.argsort(X[:, j]); xs, ys, ws = X[o, j], y[o], w[o]
        pos_left = np.r_[0, np.cumsum(ws * (ys > 0))]          # weight of positives at or left of each cut
        neg_left = np.r_[0, np.cumsum(ws * (ys < 0))]
        pos_tot, neg_tot = pos_left[-1], neg_left[-1]
        err_plus = pos_left + (neg_tot - neg_left)             # predict -1 on the left, +1 on the right
        err_minus = ...
        for s, err in [(1, err_plus), (-1, err_minus)]:
            k = int(np.argmin(err))
            t = xs[0] - 1 if k == 0 else (xs[-1] + 1 if k == len(xs) else (xs[k - 1] + xs[k]) / 2)
            if err[k] < best[3]:
                best = (j, t, s, err[k])
    return best

stump_pred = lambda X, j, t, s: np.where(X[:, j] > t, s, -s)

X, y = ring(400, r); X_te, y_te = ring(3000, r)
w = np.ones(400) / 400; F_te = np.zeros(3000); ada_acc = []
for t_ in range(200):
    j, t, s, err = weighted_stump(X, y, w)
    err = max(err, 1e-10)
    alpha = ...
    h = stump_pred(X, j, t, s)
    w = ...
    w /= w.sum()
    F_te += alpha * stump_pred(X_te, j, t, s)
    if t_ + 1 in (10, 50, 200):
        ada_acc.append(np.mean(np.sign(F_te) == y_te))
j, t, s, _ = weighted_stump(X, y, np.ones(400) / 400)
stump_acc = np.mean(stump_pred(X_te, j, t, s) == y_te)
print(f"one stump: {stump_acc:.3f}; boosted after 10, 50, 200 rounds: {[round(float(a), 3) for a in ada_acc]}")

In [ ]:
import numpy as np
def test_stump():
    "weighted_stump finds a perfect stump on a separable one-feature problem"
    Xs = np.array([[0.0], [1.0], [2.0], [3.0]]); ys = np.array([-1, -1, 1, 1.0])
    j, t, s, err = weighted_stump(Xs, ys, np.ones(4) / 4)
    assert err < 1e-12 and 1 < t < 2
def test_boost():
    "single stumps are weak, but 200 rounds reach test accuracy above 0.85"
    assert stump_acc < 0.7 and ada_acc[-1] > 0.85

test_stump()
test_boost()
print("All checks passed")

### Exercise: Gradient boosting a tuning surface

Write reg_stump(X, res), returning the feature, threshold and the two leaf means that minimise squared error, using cumulative sums. Then run gradient boosting for squared loss with learning rate 0.1, recording training and test MSE every 50 rounds.

In [ ]:
def reg_stump(X, res):
    best = (0, 0.0, 0.0, 0.0, np.inf); n = len(res)
    for j in range(X.shape[1]):
        o = np.argsort(X[:, j]); xs, rs = X[o, j], res[o]
        cs = np.cumsum(rs)[:-1]; nl = np.arange(1, n)
        score = cs ** 2 / nl + (rs.sum() - cs) ** 2 / (n - nl)        # larger is better (less squared error)
        score = np.where(xs[1:] > xs[:-1], score, -np.inf)
        k = int(np.argmax(score))
        if -score[k] < best[4]:
            best = (j, (xs[k] + xs[k + 1]) / 2, cs[k] / nl[k], ..., -score[k])
    return best[:4]

def surface(n, g):
    X = g.uniform(-1, 1, (n, 3))                    # stimulus orientation, contrast, and an irrelevant feature
    f = 10 * np.exp(-(X[:, 0] / 0.4) ** 2) * (1 + X[:, 1]) + 2 * X[:, 1]
    return X, f + 1.5 * g.normal(size=n)

Xr, yr = surface(500, r); Xr_te, yr_te = surface(3000, r)
nu = 0.1
F, F_te = np.full(500, yr.mean()), np.full(3000, yr.mean())
gb_tr, gb_te = [], []
for m in range(300):
    j, t, a, b = reg_stump(Xr, yr - F)
    F += ...
    F_te += nu * np.where(Xr_te[:, j] <= t, a, b)
    if (m + 1) % 50 == 0:
        gb_tr.append(np.mean((yr - F) ** 2)); gb_te.append(np.mean((yr_te - F_te) ** 2))
print("rounds 50 to 300, training MSE:", np.round(gb_tr, 2))
print("rounds 50 to 300, test MSE:    ", np.round(gb_te, 2), f"(target variance {np.var(yr_te):.1f}, noise 2.25)")

In [ ]:
import numpy as np
def test_stump():
    "reg_stump returns the split minimising squared error on the residuals"
    Xs = np.array([[0.0, 5], [1, 4], [2, 3], [3, 2]]); rs = np.array([-1.0, -1, 1, 1])
    j, t, a, b = reg_stump(Xs, rs)
    assert abs(a + 1) < 1e-12 and abs(b - 1) < 1e-12
def test_fit():
    "300 rounds reduce test MSE well below the variance of the target"
    assert gb_te[-1] < 0.4 * np.var(yr_te)

test_stump()
test_fit()
print("All checks passed")

### Exercise: Learning rate against rounds

Split the 500 training examples into 350 for fitting and 150 for validation. For learning rates 0.05 and 0.8, boost for 400 rounds and record the validation MSE after every round. Report each curve's minimum and where it occurs.

In [ ]:
fit_i, val_i = np.arange(350), np.arange(350, 500)
val = {}
for lr_ in [0.05, 0.8]:
    F, Fv, curve = np.full(350, yr[fit_i].mean()), np.full(150, yr[fit_i].mean()), []
    for m in range(400):
        j, t, a, b = reg_stump(Xr[fit_i], yr[fit_i] - F)
        F += lr_ * np.where(Xr[fit_i][:, j] <= t, a, b)
        Fv += ...
        curve.append(np.mean((yr[val_i] - Fv) ** 2))
    val[lr_] = curve
    print(f"learning rate {lr_}: best validation MSE {min(curve):.2f} at round {int(np.argmin(curve)) + 1}; after 400 rounds {curve[-1]:.2f}")

In [ ]:
import numpy as np
def test_curves():
    "val holds a validation curve of 400 rounds for each learning rate"
    assert all(len(v) == 400 for v in val.values())
def test_small_lr():
    "the small learning rate reaches a lower minimum validation error, at a later round"
    assert min(val[0.05]) <= min(val[0.8]) and np.argmin(val[0.05]) > np.argmin(val[0.8])

test_curves()
test_small_lr()
print("All checks passed")